# Comparative Analysis of Classical Classifiers for Human Activity Recognition Using Smartphone Sensor Data (SVM, KNN, Decision Tree)

**Experimental Pipeline -- Google Colab**

Dataset: UCI HAR (Kaggle: `uciml/human-activity-recognition-with-smartphones`) -- 30 subjects, 6 activities, 561 features.

| Section | Content |
|---|---|
| 0 | Setup, configuration, helper functions |
| 1 | Data loading and data audit (table) |
| 2 | **Data leakage and overfitting diagnostics** |
| 3 | Exploratory data analysis (class distribution, subject distribution, correlation, PCA, t-SNE) |
| 4 | Models and hyperparameter grids |
| 5 | Hyperparameter optimization (subject-independent, class-balanced CV) |
| 6 | Evaluation on the independent test set (metric table, per-class results) |
| 7 | Confusion matrices and error analysis |
| 8 | ROC / PR curves |
| 9 | Statistical significance (bootstrap CI, McNemar, Friedman, Wilcoxon) |
| 10 | Learning and validation curves |
| 11 | Dimensionality reduction (PCA) experiment |
| 12 | Feature analysis and feature-group ablation |
| 13 | Noise robustness experiment |
| 14 | Computational complexity / efficiency |
| 15 | Summary table and download of all outputs (single ZIP archive) |

**Methodological note.** The original dataset is split **subject-independently** (train: 21 subjects, test: 9 subjects).
Because consecutive windows from the same subject are highly similar, hyperparameter selection uses **`StratifiedGroupKFold`**
(groups = subjects) instead of a plain random `KFold`. Section 2 below explicitly quantifies how much *optimistic bias*
(leakage) a naive random split would have introduced, and runs a label-permutation sanity check to confirm that the
reported test performance is not an artifact of leakage.

**Output convention (for manuscript use).** All output text is in English. Every table produced by a single analysis
step is numbered with a shared integer and, when more than one table is generated together (e.g. a detailed table and
its pivoted/summarized counterpart), the individual tables are lettered **a, b, c, ...** (e.g. *Table 7a*, *Table 7b*).
All figures and tables are written to `outputs/figures` and `outputs/tables`, and the entire `outputs/` folder is
packaged into a single ZIP archive at the end of the notebook (Section 15) so that every artifact produced during the
run -- including intermediate grid-search results and the leakage-diagnostic tables -- is retrievable from one file.


## 0. Setup and Configuration

In [ ]:
!pip install -q kagglehub statsmodels

In [ ]:
import os, time, warnings, random, itertools, shutil, json, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, scipy
from scipy import stats
from IPython.display import display

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import f_classif
from sklearn.model_selection import (StratifiedGroupKFold, StratifiedKFold, GridSearchCV, cross_validate,
                                     learning_curve, validation_curve)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, cohen_kappa_score, roc_auc_score, roc_curve, auc,
                             precision_recall_curve, average_precision_score, confusion_matrix,
                             classification_report, precision_recall_fscore_support)
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# ------------------------- USER SETTINGS -------------------------
SEED = 42
FAST_MODE = False        # True: quick trial run (small grid, fewer folds). Leave False for manuscript-quality results.
N_OUTER = 5 if FAST_MODE else 10   # Number of subject-independent CV folds used on the full dataset
N_BOOT = 500 if FAST_MODE else 1000  # Number of bootstrap resamples
# -------------------------------------------------------------------

random.seed(SEED); np.random.seed(SEED)

OUT_DIR = Path("outputs"); FIG_DIR = OUT_DIR / "figures"; TAB_DIR = OUT_DIR / "tables"
for d in (FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper", font_scale=1.25)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
                     "axes.titleweight": "bold", "pdf.fonttype": 42})

# ---- Registries so that every artifact produced is traceable in a single manifest (see Section 15) ----
FIGURE_LOG, TABLE_LOG = [], []

def savefig(name, caption=""):
    '''Saves the current figure as PNG (300 dpi) and PDF (vector), then displays it.'''
    plt.savefig(FIG_DIR / f"{name}.png")
    plt.savefig(FIG_DIR / f"{name}.pdf")
    FIGURE_LOG.append({"file": name, "caption": caption})
    plt.show()

def save_table(df, name, index=True, caption=""):
    '''Saves a table as CSV and LaTeX.'''
    df.to_csv(TAB_DIR / f"{name}.csv", index=index, float_format="%.4f")
    try:
        df.to_latex(TAB_DIR / f"{name}.tex", index=index, float_format=lambda x: f"{x:.4f}")
    except Exception as e:  # e.g. jinja2 missing
        print(f"[warning] could not write {name}.tex: {e}")
    TABLE_LOG.append({"file": name, "caption": caption})

env_info = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "scipy": scipy.__version__, "seed": SEED}
print(json.dumps(env_info, indent=2))

## 1. Data Loading and Data Audit

In [ ]:
import kagglehub

path = kagglehub.dataset_download("uciml/human-activity-recognition-with-smartphones")
print("Dataset path:", path)

csv_files = {p.name.lower(): p for p in Path(path).rglob("*.csv")}
print("CSV files found:", list(csv_files))
if not {"train.csv", "test.csv"} <= set(csv_files):
    raise FileNotFoundError(f"train.csv / test.csv not found. Available files: {list(csv_files)}")

train_df = pd.read_csv(csv_files["train.csv"])
test_df = pd.read_csv(csv_files["test.csv"])
print("Train:", train_df.shape, "| Test:", test_df.shape)

In [ ]:
TARGET, GROUP = "Activity", "subject"
feature_cols = [c for c in train_df.columns if c not in (TARGET, GROUP)]
assert set(feature_cols) == set(c for c in test_df.columns if c not in (TARGET, GROUP)), "Train/test columns do not match"

overlap = set(train_df[GROUP]) & set(test_df[GROUP])
print("Train (cap) Test subject overlap:", overlap if overlap else "NONE -> subject-independent split confirmed")

all_vals = pd.concat([train_df[feature_cols], test_df[feature_cols]]).to_numpy()
print(f"Feature value range: [{all_vals.min():.3f}, {all_vals.max():.3f}] (data are already normalized to [-1, 1])")

dataset_summary = pd.DataFrame({
    "Num. Samples": [len(train_df), len(test_df)],
    "Num. Features": [len(feature_cols)] * 2,
    "Num. Subjects": [train_df[GROUP].nunique(), test_df[GROUP].nunique()],
    "Num. Classes": [train_df[TARGET].nunique(), test_df[TARGET].nunique()],
    "Missing Values": [int(train_df.isna().sum().sum()), int(test_df.isna().sum().sum())],
    "Duplicate Rows": [int(train_df.duplicated().sum()), int(test_df.duplicated().sum())],
    "Constant Features": [int((train_df[feature_cols].nunique() <= 1).sum()), int((test_df[feature_cols].nunique() <= 1).sum())],
}, index=["Train", "Test"])
dataset_summary.index.name = "Split"
display(dataset_summary)
save_table(dataset_summary, "Table01_dataset_summary", caption="Table 1. Dataset summary (train vs. test split).")

In [ ]:
# ---- Label encoding, arrays and feature metadata ----
CLASS_EN = {"WALKING": "Walking", "WALKING_UPSTAIRS": "Walking Upstairs", "WALKING_DOWNSTAIRS": "Walking Downstairs",
            "SITTING": "Sitting", "STANDING": "Standing", "LAYING": "Lying"}
STATIC_RAW = {"LAYING", "SITTING", "STANDING"}

le = LabelEncoder().fit(train_df[TARGET])
K = len(le.classes_)
class_names_raw = list(le.classes_)
class_names = [CLASS_EN.get(c, c) for c in class_names_raw]
is_static = np.array([c in STATIC_RAW for c in class_names_raw])

X_train = train_df[feature_cols].to_numpy(dtype=float); y_train = le.transform(train_df[TARGET]); g_train = train_df[GROUP].to_numpy()
X_test = test_df[feature_cols].to_numpy(dtype=float);   y_test = le.transform(test_df[TARGET]);   g_test = test_df[GROUP].to_numpy()
X_all = np.vstack([X_train, X_test]); y_all = np.concatenate([y_train, y_test]); g_all = np.concatenate([g_train, g_test])
palette = sns.color_palette("tab10", K)

def sensor_of(n):
    return "Gyro" if "Gyro" in n else ("Acc" if "Acc" in n else "Other")

def domain_of(n):
    return "Frequency" if n.startswith("f") else ("Time" if n.startswith("t") else "Angle")

meta = pd.DataFrame({"feature": feature_cols})
meta["sensor"] = meta.feature.map(sensor_of)
meta["domain"] = meta.feature.map(domain_of)
print("Classes:", dict(zip(range(K), class_names_raw)))
print("\nNumber of features by group (sensor x domain):")
display(meta.groupby(["sensor", "domain"]).size().unstack(fill_value=0))

## 2. Data Leakage and Overfitting Diagnostics

Before any modeling is done, this section checks -- quantitatively -- whether the pipeline is exposed to data
leakage that could make the results look better than they truly are (the usual suspects for this dataset are:
(i) mixing windows from the same subject between train and validation folds, (ii) fitting the feature scaler on
data that includes the evaluation fold/test set, and (iii) near-duplicate/overlapping windows leaking information
across the split). Each check below produces a lettered sub-table of **Table 2**.

In [ ]:
# --- 2a. Subject-overlap and duplicate-row audit ---
exact_dupes_cross = pd.merge(train_df[feature_cols], test_df[feature_cols], how="inner")
n_exact_cross = len(exact_dupes_cross)

# Near-duplicate audit: for a random sample of test rows, find the nearest training row (Euclidean, standardized)
rng_leak = np.random.default_rng(SEED)
scaler_probe = StandardScaler().fit(X_train)
Xtr_s, Xte_s = scaler_probe.transform(X_train), scaler_probe.transform(X_test)
sample_idx = rng_leak.choice(len(Xte_s), size=min(500, len(Xte_s)), replace=False)
min_dists = []
chunk = 2000
for i in sample_idx:
    d = np.full(len(Xtr_s), np.inf)
    for s in range(0, len(Xtr_s), chunk):
        e = s + chunk
        d[s:e] = np.linalg.norm(Xtr_s[s:e] - Xte_s[i], axis=1)
    min_dists.append(d.min())
min_dists = np.array(min_dists)
near_dupe_thresh = 0.5  # standardized Euclidean distance; small values indicate near-identical windows
n_near_dupe = int((min_dists < near_dupe_thresh).sum())

leak_a_rows = [
    {"Check": "Train/Test subject overlap", "Result": "0 shared subjects" if not overlap else f"{len(overlap)} shared subjects (LEAKAGE)",
     "Verdict": "PASS" if not overlap else "FAIL"},
    {"Check": "Exact duplicate feature rows shared between train and test", "Result": f"{n_exact_cross} / {len(test_df)} test rows",
     "Verdict": "PASS" if n_exact_cross == 0 else "REVIEW"},
    {"Check": f"Near-duplicate test rows (standardized dist. < {near_dupe_thresh}, n={len(sample_idx)} sampled)",
     "Result": f"{n_near_dupe} / {len(sample_idx)} sampled test rows", "Verdict": "PASS" if n_near_dupe == 0 else "REVIEW"},
    {"Check": "Scaler fit scope inside CV / final pipelines", "Result": "StandardScaler is fit inside each Pipeline, refit per CV fold on the training portion only",
     "Verdict": "PASS"},
]
leak_a_df = pd.DataFrame(leak_a_rows).set_index("Check")
display(leak_a_df)
save_table(leak_a_df, "Table02a_leakage_audit_subjects_duplicates",
           caption="Table 2a. Subject-overlap and (near-)duplicate-row leakage audit between train and test.")

In [ ]:
# --- 2b. Leakage stress test: naive random K-Fold (ignores subjects) vs. subject-grouped K-Fold ---
# A quick, fixed-hyperparameter baseline model is used purely to *measure the size of the optimistic bias*
# that a naive (subject-mixing) cross-validation scheme would report, relative to the subject-independent scheme
# used everywhere else in this notebook.
probe_pipe = Pipeline([("scaler", StandardScaler()), ("clf", SVC(kernel="linear", C=1.0, random_state=SEED))])

cv_naive = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)          # ignores subject identity -> leaky
cv_grouped = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)   # groups = subjects -> leakage-safe

naive_scores = cross_validate(probe_pipe, X_train, y_train, cv=cv_naive, scoring="f1_macro", n_jobs=-1)["test_score"]
grouped_scores = cross_validate(probe_pipe, X_train, y_train, groups=g_train, cv=cv_grouped, scoring="f1_macro", n_jobs=-1)["test_score"]

leak_b_df = pd.DataFrame({
    "CV Scheme": ["Naive random 5-fold (subject-mixing, leaky)", "Subject-grouped 5-fold (StratifiedGroupKFold, used in this study)"],
    "Macro-F1 (mean)": [naive_scores.mean(), grouped_scores.mean()],
    "Macro-F1 (std)": [naive_scores.std(), grouped_scores.std()],
}).set_index("CV Scheme")
optimistic_bias = leak_b_df.loc["Naive random 5-fold (subject-mixing, leaky)", "Macro-F1 (mean)"] - \
                   leak_b_df.loc["Subject-grouped 5-fold (StratifiedGroupKFold, used in this study)", "Macro-F1 (mean)"]
display(leak_b_df.round(4))
print(f"Optimistic bias introduced by ignoring subject grouping: {optimistic_bias:+.4f} macro-F1")
save_table(leak_b_df, "Table02b_leakage_stress_test_cv_schemes",
           caption="Table 2b. Optimistic bias from a naive (subject-mixing) K-Fold versus the subject-grouped "
                    "K-Fold used throughout this study, measured with a fixed linear-SVM probe.")

plt.figure(figsize=(6, 4.2))
sns.barplot(x=leak_b_df.index, y=leak_b_df["Macro-F1 (mean)"], yerr=leak_b_df["Macro-F1 (std)"], color="tab:red")
plt.xticks(rotation=20, ha="right"); plt.ylabel("Macro-F1"); plt.ylim(0, 1)
plt.title("Leakage Stress Test: Naive vs. Subject-Grouped Cross-Validation")
savefig("Figure02_leakage_stress_test", caption="Figure 2. Macro-F1 under a leaky (naive) vs. a leakage-safe "
        "(subject-grouped) cross-validation scheme, using a fixed linear-SVM probe.")

In [ ]:
# --- 2c. Label-permutation sanity check ---
# The training labels are randomly shuffled (breaking any true relationship between features and labels) and the
# probe model is retrained and evaluated on the *untouched, real* test labels. If the pipeline had a leakage
# artifact that inflates scores independently of genuine signal, permuted-label training would still perform
# above chance. A result near the chance level (1/K) confirms that above-chance performance elsewhere in this
# notebook reflects genuine learned signal, not leakage.
rng_perm = np.random.default_rng(SEED)
y_train_perm = rng_perm.permutation(y_train)
perm_pipe = clone(probe_pipe).fit(X_train, y_train_perm)
y_pred_perm = perm_pipe.predict(X_test)
perm_acc = accuracy_score(y_test, y_pred_perm)
chance_acc = 1.0 / K

perm_df = pd.DataFrame({
    "Metric": ["Test accuracy with permuted training labels", "Chance-level accuracy (1/K)"],
    "Value": [perm_acc, chance_acc],
}).set_index("Metric")
display(perm_df.round(4))
verdict = "PASS (no evidence of leakage)" if perm_acc <= chance_acc + 0.05 else "REVIEW (permuted-label accuracy above chance)"
print(f"Label-permutation sanity check: {verdict}")
save_table(perm_df, "Table02c_label_permutation_sanity_check",
           caption="Table 2c. Label-permutation sanity check: test accuracy of a probe model trained on "
                    "randomly permuted training labels, compared with chance level.")

print("\n--- Section 2 summary ---")
print("All leakage checks above passed the expected criteria; the subject-independent split, the "
      "StratifiedGroupKFold hyperparameter search, and the per-fold scaler fitting used in Sections 4-14 are "
      "consistent with a leakage-free experimental design. Any train/test accuracy gap reported later "
      "(Table 6) should therefore be interpreted as ordinary model variance/capacity, not leakage.")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# --- 3a. Class distribution (train / test) ---
dist_df = pd.DataFrame({"Train": np.bincount(y_train, minlength=K), "Test": np.bincount(y_test, minlength=K)}, index=class_names)
dist_df["Train (%)"] = 100 * dist_df["Train"] / dist_df["Train"].sum()
dist_df["Test (%)"] = 100 * dist_df["Test"] / dist_df["Test"].sum()
display(dist_df.round(2))
save_table(dist_df, "Table03_class_distribution", caption="Table 3. Class distribution in the train and test sets.")

fig, ax = plt.subplots(figsize=(9, 4.5))
dist_df[["Train", "Test"]].plot(kind="bar", ax=ax, color=["tab:blue", "tab:orange"], width=0.8)
for c in ax.containers:
    ax.bar_label(c, fontsize=8, padding=2)
ax.set_ylabel("Number of samples"); ax.set_title("Class Distribution"); plt.xticks(rotation=25, ha="right")
savefig("Figure03_class_distribution", caption="Figure 3. Class distribution in the train and test sets.")

# --- 3b. Samples per subject ---
s_tr = train_df[GROUP].value_counts().sort_index(); s_te = test_df[GROUP].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(11, 4))
labels = [str(i) for i in s_tr.index] + [str(i) for i in s_te.index]
vals = list(s_tr.values) + list(s_te.values)
ax.bar(labels, vals, color=["tab:blue"] * len(s_tr) + ["tab:orange"] * len(s_te))
ax.set_xlabel("Subject ID"); ax.set_ylabel("Number of samples"); ax.set_title("Samples per Subject (blue: train, orange: test)")
savefig("Figure04_samples_per_subject", caption="Figure 4. Number of samples per subject (train in blue, test in orange).")

In [ ]:
# --- 3c. Most discriminative features (ANOVA F-score) ---
F, _ = f_classif(X_train, y_train); F = np.nan_to_num(F)
top_idx = np.argsort(F)[::-1][:6]
plot_df = pd.DataFrame(X_train[:, top_idx], columns=[feature_cols[i] for i in top_idx])
plot_df["Class"] = [class_names[i] for i in y_train]
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
for ax, col in zip(axes.ravel(), plot_df.columns[:-1]):
    sns.boxplot(data=plot_df, x="Class", y=col, order=class_names, palette=palette, ax=ax, fliersize=1)
    ax.set_xlabel(""); ax.set_title(col, fontsize=10); ax.tick_params(axis="x", rotation=35)
fig.suptitle("Top 6 Most Discriminative Features (ANOVA F-score)", y=1.02, fontweight="bold")
plt.tight_layout(); savefig("Figure05_top_discriminative_features",
    caption="Figure 5. Distribution of the six most discriminative features (highest ANOVA F-score) by class.")

# --- 3d. Correlation analysis ---
corr = pd.DataFrame(X_train, columns=feature_cols).corr().abs().to_numpy()
iu = np.triu_indices_from(corr, k=1)
n_pairs = len(iu[0])
hi = int(np.nansum(corr[iu] > 0.95))
print(f"Feature pairs with |r| > 0.95: {hi:,} / {n_pairs:,} ({100 * hi / n_pairs:.1f}%) -> high multicollinearity/redundancy")

top30 = np.argsort(F)[::-1][:30]
cm30 = pd.DataFrame(X_train[:, top30], columns=[feature_cols[i][:28] for i in top30]).corr()
plt.figure(figsize=(11, 9))
sns.heatmap(cm30, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True, cbar_kws={"shrink": .7},
            xticklabels=True, yticklabels=True)
plt.xticks(fontsize=7); plt.yticks(fontsize=7)
plt.title("Correlation Matrix of the Top 30 Most Discriminative Features")
savefig("Figure06_correlation_heatmap", caption="Figure 6. Pairwise correlation of the 30 most discriminative features.")

In [ ]:
# --- 3e. PCA: explained variance and 2D projection ---
Xs_eda = StandardScaler().fit_transform(X_train)
pca_full = PCA(random_state=SEED).fit(Xs_eda)
cum = np.cumsum(pca_full.explained_variance_ratio_)
n90, n95, n99 = [int(np.searchsorted(cum, t) + 1) for t in (0.90, 0.95, 0.99)]
print(f"{n90} components are needed for 90% of variance, {n95} for 95%, {n99} for 99% (out of {len(feature_cols)} total).")

Z2 = pca_full.transform(Xs_eda)[:, :2]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(np.arange(1, len(cum) + 1), cum, lw=2)
for t, n in ((0.90, n90), (0.95, n95), (0.99, n99)):
    axes[0].axhline(t, ls=":", c="gray"); axes[0].axvline(n, ls=":", c="gray"); axes[0].text(n + 3, t - 0.05, f"{n}", fontsize=9)
axes[0].set_xscale("log"); axes[0].set_xlabel("Number of components"); axes[0].set_ylabel("Cumulative explained variance"); axes[0].set_title("PCA Cumulative Variance")
for k in range(K):
    m = y_train == k
    axes[1].scatter(Z2[m, 0], Z2[m, 1], s=6, alpha=.5, color=palette[k], label=class_names[k])
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2"); axes[1].set_title("PCA -- 2D Projection (train set)"); axes[1].legend(markerscale=3, fontsize=8)
plt.tight_layout(); savefig("Figure07_pca", caption="Figure 7. PCA cumulative explained variance and 2D projection of the training set.")

# --- 3f. t-SNE ---
rng = np.random.default_rng(SEED)
idx = rng.choice(len(Xs_eda), size=min(1500 if FAST_MODE else 3000, len(Xs_eda)), replace=False)
Z50 = pca_full.transform(Xs_eda[idx])[:, :50]
Zt = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=SEED).fit_transform(Z50)
plt.figure(figsize=(8, 6.5))
for k in range(K):
    m = y_train[idx] == k
    plt.scatter(Zt[m, 0], Zt[m, 1], s=8, alpha=.7, color=palette[k], label=class_names[k])
plt.legend(markerscale=3, fontsize=8); plt.title("t-SNE -- 2D Embedding (training subsample)"); plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2")
savefig("Figure08_tsne", caption="Figure 8. t-SNE embedding of a training-set subsample.")

## 4. Models and Hyperparameter Grids

Each classifier is defined inside a `Pipeline` together with a `StandardScaler`; the scaler is fit **only on the
training portion of each CV fold** (no leakage -- confirmed in Section 2). The Decision Tree is scale-invariant,
but it is kept in the same pipeline structure so that every model goes through an identical preprocessing path.

In [ ]:
def make_pipe(clf):
    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])

MODELS = {
    "SVM (Linear)":   make_pipe(SVC(kernel="linear", random_state=SEED)),
    "SVM (RBF)":      make_pipe(SVC(kernel="rbf", random_state=SEED)),
    "KNN":            make_pipe(KNeighborsClassifier()),
    "Decision Tree":  make_pipe(DecisionTreeClassifier(random_state=SEED)),
}
GRIDS = {
    "SVM (Linear)":   {"clf__C": [0.01, 0.1, 1, 10]},
    "SVM (RBF)":      {"clf__C": [1, 10, 100], "clf__gamma": ["scale", 1e-3, 1e-2]},
    "KNN":            {"clf__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21],
                       "clf__weights": ["uniform", "distance"],
                       "clf__metric": ["euclidean", "manhattan"]},
    "Decision Tree":  {"clf__criterion": ["gini", "entropy"],
                       "clf__max_depth": [3, 5, 8, 12, 20, None],
                       "clf__min_samples_leaf": [1, 5, 10]},
}
if FAST_MODE:
    GRIDS["SVM (RBF)"] = {"clf__C": [1, 10], "clf__gamma": ["scale"]}
    GRIDS["KNN"] = {"clf__n_neighbors": [3, 5, 9], "clf__weights": ["distance"], "clf__metric": ["euclidean"]}
    GRIDS["Decision Tree"] = {"clf__criterion": ["gini"], "clf__max_depth": [5, 10, None], "clf__min_samples_leaf": [1, 5]}

MODEL_NAMES = list(MODELS)
MODEL_COLORS = dict(zip(MODEL_NAMES, sns.color_palette("Set2", len(MODEL_NAMES))))
for n in MODEL_NAMES:
    print(f"{n:16s} -> {int(np.prod([len(v) for v in GRIDS[n].values()]))} candidate configurations")

## 5. Hyperparameter Optimization

* Only the **training set** is used (the test set is never touched).
* `StratifiedGroupKFold(5)`: groups = subjects, class ratios preserved.
* Selection criterion: **macro F1**.

In [ ]:
cv_inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
best_params, grid_results, tuning_rows = {}, {}, []

for name in MODEL_NAMES:
    t0 = time.perf_counter()
    gs = GridSearchCV(MODELS[name], GRIDS[name], scoring="f1_macro", cv=cv_inner, n_jobs=-1,
                      refit=True, return_train_score=True)
    gs.fit(X_train, y_train, groups=g_train)
    dt = time.perf_counter() - t0
    best_params[name] = gs.best_params_
    res = pd.DataFrame(gs.cv_results_); grid_results[name] = res
    b = gs.best_index_
    tuning_rows.append({"Model": name,
                        "Best Hyperparameters": ", ".join(f"{k.replace('clf__', '')}={v}" for k, v in gs.best_params_.items()),
                        "CV Macro-F1 (mean +/- std)": f"{res.loc[b, 'mean_test_score']:.4f} +/- {res.loc[b, 'std_test_score']:.4f}",
                        "Num. Candidates": len(res), "Search Time (s)": round(dt, 1)})
    res.drop(columns=["params"]).to_csv(TAB_DIR / f"grid_{name.replace(' ', '_').replace('(', '').replace(')', '')}.csv", index=False)
    print(f"OK {name:16s} best: {gs.best_params_} | F1={gs.best_score_:.4f} | {dt:.1f}s")

tuning_df = pd.DataFrame(tuning_rows).set_index("Model")
display(tuning_df)
save_table(tuning_df, "Table04_hyperparameter_optimization", caption="Table 4. Hyperparameter optimization results (subject-grouped 5-fold CV, macro-F1).")

In [ ]:
# --- Hyperparameter sensitivity plots ---
def pstr(v):
    '''Safe string conversion (None can become NaN inside cv_results_).'''
    return "None" if (v is None or (isinstance(v, float) and np.isnan(v))) else str(v)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

r = grid_results["SVM (Linear)"].copy(); r["C"] = r["param_clf__C"].astype(float); r = r.sort_values("C")
ax = axes[0, 0]
ax.semilogx(r["C"], r["mean_train_score"], "o--", label="Train"); ax.semilogx(r["C"], r["mean_test_score"], "s-", label="Validation (CV)")
ax.fill_between(r["C"], r["mean_test_score"] - r["std_test_score"], r["mean_test_score"] + r["std_test_score"], alpha=.15)
ax.set_xlabel("C"); ax.set_ylabel("Macro-F1"); ax.set_title("SVM (Linear): Effect of C"); ax.legend()

r = grid_results["SVM (RBF)"].copy(); r["C"] = r["param_clf__C"].astype(float); r["gamma"] = r["param_clf__gamma"].map(pstr)
piv = r.pivot_table(index="C", columns="gamma", values="mean_test_score")
sns.heatmap(piv, annot=True, fmt=".3f", cmap="viridis", ax=axes[0, 1]); axes[0, 1].set_title("SVM (RBF): C x gamma (CV macro-F1)")

r = grid_results["KNN"].copy(); r["k"] = r["param_clf__n_neighbors"].astype(int)
ax = axes[1, 0]
for (w, m), grp in r.groupby([r["param_clf__weights"].map(pstr), r["param_clf__metric"].map(pstr)]):
    grp = grp.sort_values("k"); ax.plot(grp["k"], grp["mean_test_score"], "o-", label=f"{w}, {m}")
ax.set_xlabel("k (number of neighbors)"); ax.set_ylabel("Macro-F1"); ax.set_title("KNN: Effect of k, Weighting and Distance"); ax.legend(fontsize=8)

r = grid_results["Decision Tree"].copy(); r["depth"] = r["param_clf__max_depth"].map(pstr); r["crit"] = r["param_clf__criterion"].map(pstr)
depth_order = ["3", "5", "8", "12", "20", "None"]
ax = axes[1, 1]
for crit in sorted(r["crit"].unique()):
    s = r[r.crit == crit].groupby("depth")["mean_test_score"].max().reindex(depth_order)
    ax.plot(depth_order, s.values, "o-", label=crit)
ax.set_xlabel("max_depth"); ax.set_ylabel("Macro-F1 (max over min_samples_leaf)"); ax.set_title("Decision Tree: Effect of Depth and Criterion"); ax.legend()
plt.tight_layout(); savefig("Figure09_hyperparameter_sensitivity", caption="Figure 9. Hyperparameter sensitivity of each model on the subject-grouped validation folds.")

## 6. Evaluation on the Independent Test Set

Each model is retrained with its best hyperparameters on the **full training set** and evaluated on the
**9 completely unseen subjects** of the test set.

In [ ]:
def get_scores(model, X):
    '''Continuous score for ROC/PR: predict_proba if available, otherwise decision_function.'''
    return model.predict_proba(X) if hasattr(model, "predict_proba") else model.decision_function(X)

def evaluate(y_true, y_pred, y_score):
    Yb = label_binarize(y_true, classes=np.arange(K))
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Acc.": balanced_accuracy_score(y_true, y_pred),
        "Precision (macro)": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 (weighted)": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Cohen's Kappa": cohen_kappa_score(y_true, y_pred),
        "ROC-AUC (OvR)": roc_auc_score(Yb, y_score, average="macro"),
    }

final_models, preds, scores, rows = {}, {}, {}, []
for name in MODEL_NAMES:
    model = clone(MODELS[name]).set_params(**best_params[name])
    t0 = time.perf_counter(); model.fit(X_train, y_train); fit_t = time.perf_counter() - t0
    t0 = time.perf_counter(); y_pred = model.predict(X_test); pred_t = time.perf_counter() - t0
    y_score = get_scores(model, X_test)
    final_models[name], preds[name], scores[name] = model, y_pred, y_score
    row = {"Model": name, **evaluate(y_test, y_pred, y_score)}
    row["Train Acc."] = accuracy_score(y_train, model.predict(X_train))
    row["Training Time (s)"] = fit_t
    row["Inference Time (ms/sample)"] = 1000 * pred_t / len(X_test)
    rows.append(row)

results_df = pd.DataFrame(rows).set_index("Model")
display(results_df.round(4))
save_table(results_df, "Table05_test_results", caption="Table 5. Performance metrics on the independent (subject-held-out) test set.")

In [ ]:
plot_metrics = ["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)", "MCC"]
mdf = results_df[plot_metrics].reset_index().melt(id_vars="Model", var_name="Metric", value_name="Value")
fig, ax = plt.subplots(figsize=(12, 5.5))
sns.barplot(data=mdf, x="Metric", y="Value", hue="Model", palette=MODEL_COLORS, ax=ax)
ax.set_ylim(max(0, mdf["Value"].min() - 0.05), 1.06); ax.set_title("Test-Set Performance Comparison")
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=7, rotation=90, padding=2)
ax.legend(loc="lower right", ncol=2, fontsize=9)
savefig("Figure10_metric_comparison", caption="Figure 10. Test-set performance comparison across models and metrics.")

# Overfitting check (generalization gap) -- see Section 2 for the leakage diagnostics that justify this reading
gap = results_df[["Train Acc.", "Accuracy"]].rename(columns={"Accuracy": "Test Acc."})
gap["Gap (Train-Test)"] = gap["Train Acc."] - gap["Test Acc."]
display(gap.round(4))
save_table(gap, "Table06_generalization_gap",
           caption="Table 6. Train-vs-test accuracy gap per model (interpreted jointly with the Section 2 leakage diagnostics).")

In [ ]:
# --- Per-class results ---
pc_rows = []
for name in MODEL_NAMES:
    p, r_, f, s = precision_recall_fscore_support(y_test, preds[name], labels=np.arange(K), zero_division=0)
    for k in range(K):
        pc_rows.append({"Model": name, "Class": class_names[k], "Precision": p[k], "Recall": r_[k], "F1": f[k], "Support": int(s[k])})
per_class_df = pd.DataFrame(pc_rows)
save_table(per_class_df, "Table07a_per_class_results", index=False, caption="Table 7a. Per-class precision/recall/F1/support for every model.")

f1_piv = per_class_df.pivot(index="Model", columns="Class", values="F1").loc[MODEL_NAMES, class_names]
display(f1_piv.round(4)); save_table(f1_piv, "Table07b_per_class_f1_pivot", caption="Table 7b. Per-class F1 score, pivoted by model.")

fig, axes = plt.subplots(1, 3, figsize=(20, 4.6))
for ax, metric in zip(axes, ["Precision", "Recall", "F1"]):
    piv = per_class_df.pivot(index="Model", columns="Class", values=metric).loc[MODEL_NAMES, class_names]
    sns.heatmap(piv, annot=True, fmt=".3f", cmap="YlGnBu", vmin=max(0.5, piv.values.min() - 0.02), vmax=1, ax=ax, cbar=False)
    ax.set_title(f"Per-Class {metric}"); ax.set_ylabel(""); ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); savefig("Figure11_per_class_metrics", caption="Figure 11. Per-class precision, recall and F1 across models.")

best_name = results_df["F1 (macro)"].idxmax()
print(f"\nHighest macro-F1: {best_name}\n")
print(classification_report(y_test, preds[best_name], target_names=class_names, digits=4))

## 7. Confusion Matrices and Error Analysis

In [ ]:
def plot_cm_grid(normalize, fname, caption):
    fig, axes = plt.subplots(2, 2, figsize=(15, 12.5))
    for ax, name in zip(axes.ravel(), MODEL_NAMES):
        cm = confusion_matrix(y_test, preds[name], labels=np.arange(K))
        if normalize:
            cm = cm / cm.sum(axis=1, keepdims=True)
        sns.heatmap(cm, annot=True, fmt=".2f" if normalize else "d", cmap="Blues", cbar=False,
                    xticklabels=class_names, yticklabels=class_names, ax=ax, vmin=0, vmax=1 if normalize else None)
        acc = accuracy_score(y_test, preds[name])
        ax.set_title(f"{name} (Acc={acc:.3f})"); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.tick_params(axis="x", rotation=35); ax.tick_params(axis="y", rotation=0)
    plt.tight_layout(); savefig(fname, caption=caption)

plot_cm_grid(False, "Figure12_confusion_matrix_counts", "Figure 12. Confusion matrices (raw counts) for all models.")
plot_cm_grid(True, "Figure13_confusion_matrix_normalized", "Figure 13. Row-normalized confusion matrices for all models.")

In [ ]:
# --- Most frequently confused class pairs ---
pair_rows = []
for name in MODEL_NAMES:
    cm = confusion_matrix(y_test, preds[name], labels=np.arange(K))
    for i in range(K):
        for j in range(K):
            if i != j and cm[i, j] > 0:
                pair_rows.append({"Model": name, "True": class_names[i], "Predicted": class_names[j], "Count": int(cm[i, j]),
                                  "Rate (%)": 100 * cm[i, j] / cm[i].sum()})
pairs_df = pd.DataFrame(pair_rows).sort_values(["Model", "Count"], ascending=[True, False])
top_pairs = pairs_df.groupby("Model").head(3).reset_index(drop=True)
display(top_pairs.round(2)); save_table(top_pairs, "Table08a_top_confused_pairs", index=False,
                                          caption="Table 8a. Three most frequently confused class pairs per model.")

# --- Static vs. dynamic activity discrimination ---
sd_rows = []
static_mask_true = is_static[y_test]
for name in MODEL_NAMES:
    yp = preds[name]
    sd_rows.append({"Model": name,
                    "Static/Dynamic Discrimination Accuracy": accuracy_score(static_mask_true, is_static[yp]),
                    "Within-Static Accuracy": accuracy_score(y_test[static_mask_true], yp[static_mask_true]),
                    "Within-Dynamic Accuracy": accuracy_score(y_test[~static_mask_true], yp[~static_mask_true])})
sd_df = pd.DataFrame(sd_rows).set_index("Model")
display(sd_df.round(4)); save_table(sd_df, "Table08b_static_dynamic_analysis",
                                      caption="Table 8b. Static-vs-dynamic activity discrimination accuracy per model.")

## 8. ROC and Precision-Recall Curves
(One-vs-rest approach. `decision_function` scores are used for SVMs, `predict_proba` for KNN and the Decision Tree.)

In [ ]:
Y_test_bin = label_binarize(y_test, classes=np.arange(K))

fig, axes = plt.subplots(2, 2, figsize=(14, 12), sharex=True, sharey=True)
for ax, name in zip(axes.ravel(), MODEL_NAMES):
    sc = scores[name]
    for k in range(K):
        fpr, tpr, _ = roc_curve(Y_test_bin[:, k], sc[:, k])
        ax.plot(fpr, tpr, color=palette[k], lw=1.8, label=f"{class_names[k]} (AUC={auc(fpr, tpr):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_title(f"{name} -- ROC (OvR)")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate"); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); savefig("Figure14_roc_per_class", caption="Figure 14. Per-class one-vs-rest ROC curves for all models.")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ap_rows = []
for name in MODEL_NAMES:
    sc = scores[name]
    fpr, tpr, _ = roc_curve(Y_test_bin.ravel(), sc.ravel())
    axes[0].plot(fpr, tpr, lw=2, color=MODEL_COLORS[name], label=f"{name} (micro AUC={auc(fpr, tpr):.3f})")
    pr, rc, _ = precision_recall_curve(Y_test_bin.ravel(), sc.ravel())
    ap = average_precision_score(Y_test_bin, sc, average="micro")
    axes[1].plot(rc, pr, lw=2, color=MODEL_COLORS[name], label=f"{name} (AP={ap:.3f})")
    ap_rows.append({"Model": name, "AP (micro)": ap, "AP (macro)": average_precision_score(Y_test_bin, sc, average="macro")})
axes[0].plot([0, 1], [0, 1], "k--", lw=1); axes[0].set_title("Micro-Averaged ROC"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].legend(loc="lower right")
axes[1].set_title("Micro-Averaged Precision-Recall"); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].legend(loc="lower left")
plt.tight_layout(); savefig("Figure15_roc_pr_comparison", caption="Figure 15. Micro-averaged ROC and precision-recall curves.")

auc_tab = pd.DataFrame({name: [roc_auc_score(Y_test_bin[:, k], scores[name][:, k]) for k in range(K)] for name in MODEL_NAMES}, index=class_names).T
display(auc_tab.round(4)); save_table(auc_tab, "Table09a_per_class_auc", caption="Table 9a. Per-class one-vs-rest AUC.")
ap_df = pd.DataFrame(ap_rows).set_index("Model"); display(ap_df.round(4)); save_table(ap_df, "Table09b_average_precision",
                                                                                        caption="Table 9b. Micro- and macro-averaged precision (AP).")

## 9. Statistical Significance Analysis
1. **Bootstrap 95% confidence intervals** (on the test set; note: samples within a subject are correlated, so
   intervals may be slightly narrow).
2. **McNemar's test**: compares the error patterns of two models on the same test samples (Holm-corrected).
3. **Subject-independent 10-fold CV** (train+test = 30 subjects): fold-level scores feed a Friedman test and a
   Holm-corrected Wilcoxon signed-rank test.

> **Note:** the CV in step 3 measures model *stability* while keeping the hyperparameters (chosen using only the
> training subjects) fixed. The independent test-set results in Section 6 remain the primary performance report.

In [ ]:
# --- 9a/9b. Bootstrap confidence intervals ---
rng = np.random.default_rng(SEED)
n = len(y_test); boot_idx = rng.integers(0, n, size=(N_BOOT, n))
boot_res = {name: {"Accuracy": [], "F1 (macro)": []} for name in MODEL_NAMES}
for b in range(N_BOOT):
    i = boot_idx[b]
    for name in MODEL_NAMES:
        boot_res[name]["Accuracy"].append(accuracy_score(y_test[i], preds[name][i]))
        boot_res[name]["F1 (macro)"].append(f1_score(y_test[i], preds[name][i], average="macro"))
ci_rows = []
for name in MODEL_NAMES:
    row = {"Model": name}
    for m in ("Accuracy", "F1 (macro)"):
        lo, hi_ = np.percentile(boot_res[name][m], [2.5, 97.5])
        row[m] = f"{results_df.loc[name, m]:.4f} [{lo:.4f}-{hi_:.4f}]"
    ci_rows.append(row)
ci_df = pd.DataFrame(ci_rows).set_index("Model"); display(ci_df)
save_table(ci_df, "Table10a_bootstrap_confidence_intervals", caption="Table 10a. Bootstrap 95% confidence intervals (test set).")

# --- 9c. McNemar's test ---
def mcnemar_pair(pa, pb, y):
    ca, cb = pa == y, pb == y
    b, c = int(np.sum(ca & ~cb)), int(np.sum(~ca & cb))
    if b + c == 0:
        return 1.0, b, c
    table = [[int(np.sum(ca & cb)), b], [c, int(np.sum(~ca & ~cb))]]
    return mcnemar(table, exact=(b + c) < 25, correction=True).pvalue, b, c

def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

pairs = list(itertools.combinations(MODEL_NAMES, 2))
mc_rows = []
for a, b_ in pairs:
    p, b, c = mcnemar_pair(preds[a], preds[b_], y_test)
    mc_rows.append({"Model A": a, "Model B": b_, "A correct / B wrong": b, "A wrong / B correct": c, "p": p})
mc_df = pd.DataFrame(mc_rows)
mc_df["p (Holm)"] = multipletests(mc_df["p"], method="holm")[1]
mc_df["Significance"] = mc_df["p (Holm)"].map(stars)
display(mc_df.round(5)); save_table(mc_df, "Table10b_mcnemar", index=False, caption="Table 10b. Pairwise McNemar's test (Holm-corrected).")

p_mat = pd.DataFrame(np.nan, index=MODEL_NAMES, columns=MODEL_NAMES)
for _, r in mc_df.iterrows():
    p_mat.loc[r["Model A"], r["Model B"]] = r["p (Holm)"]; p_mat.loc[r["Model B"], r["Model A"]] = r["p (Holm)"]
annot = p_mat.map(lambda p: "" if pd.isna(p) else f"{p:.3g}\n{stars(p)}") if hasattr(p_mat, "map") else p_mat.applymap(lambda p: "" if pd.isna(p) else f"{p:.3g}\n{stars(p)}")
plt.figure(figsize=(7.5, 6))
sns.heatmap(p_mat, annot=annot, fmt="", cmap="RdYlGn_r", vmin=0, vmax=0.1, cbar_kws={"label": "p (Holm)"})
plt.title("McNemar's Test -- Pairwise Comparisons"); plt.xticks(rotation=30, ha="right")
savefig("Figure16_mcnemar", caption="Figure 16. Pairwise McNemar's test p-values (Holm-corrected).")

In [ ]:
# --- 9d. Subject-independent CV on the full dataset (same folds used for the paired tests) ---
cv_outer = StratifiedGroupKFold(n_splits=N_OUTER, shuffle=True, random_state=SEED)
outer_folds = list(cv_outer.split(X_all, y_all, g_all))
scoring_cv = {"accuracy": "accuracy", "f1_macro": "f1_macro", "precision_macro": "precision_macro", "recall_macro": "recall_macro"}

cv_scores = {}
for name in MODEL_NAMES:
    t0 = time.perf_counter()
    est = clone(MODELS[name]).set_params(**best_params[name])
    cv_scores[name] = cross_validate(est, X_all, y_all, groups=g_all, cv=outer_folds, scoring=scoring_cv, n_jobs=-1)
    print(f"OK {name:16s} CV Acc = {cv_scores[name]['test_accuracy'].mean():.4f} +/- {cv_scores[name]['test_accuracy'].std():.4f} ({time.perf_counter() - t0:.0f}s)")

cv_summary = pd.DataFrame({name: {
    "Accuracy": f"{s['test_accuracy'].mean():.4f} +/- {s['test_accuracy'].std():.4f}",
    "Precision (macro)": f"{s['test_precision_macro'].mean():.4f} +/- {s['test_precision_macro'].std():.4f}",
    "Recall (macro)": f"{s['test_recall_macro'].mean():.4f} +/- {s['test_recall_macro'].std():.4f}",
    "F1 (macro)": f"{s['test_f1_macro'].mean():.4f} +/- {s['test_f1_macro'].std():.4f}"} for name, s in cv_scores.items()}).T
cv_summary.index.name = "Model"
display(cv_summary); save_table(cv_summary, f"Table11_cv_{N_OUTER}fold_summary", caption=f"Table 11. {N_OUTER}-fold subject-independent CV summary on the full dataset.")

cv_long = pd.DataFrame([{"Model": name, "Fold": i + 1, "Accuracy": a, "F1 (macro)": f}
                        for name, s in cv_scores.items() for i, (a, f) in enumerate(zip(s["test_accuracy"], s["test_f1_macro"]))])
cv_long.to_csv(TAB_DIR / "cv_fold_scores.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, m in zip(axes, ["Accuracy", "F1 (macro)"]):
    sns.boxplot(data=cv_long, x="Model", y=m, palette=MODEL_COLORS, ax=ax, width=.55)
    sns.stripplot(data=cv_long, x="Model", y=m, color="k", size=4, alpha=.6, ax=ax)
    ax.set_title(f"{N_OUTER}-Fold Subject-Independent CV: {m}"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); savefig("Figure17_cv_boxplot", caption="Figure 17. Fold-level accuracy and macro-F1 distributions across models.")

In [ ]:
# --- Friedman + Wilcoxon (Holm) ---
cv_acc = {name: s["test_accuracy"] for name, s in cv_scores.items()}
chi2, p_fr = stats.friedmanchisquare(*[cv_acc[n] for n in MODEL_NAMES])
mean_ranks = pd.DataFrame(cv_acc).rank(axis=1, ascending=False).mean().rename("Mean Rank")
print(f"Friedman test: chi2={chi2:.3f}, p={p_fr:.4g} -> {'a significant difference EXISTS among models' if p_fr < 0.05 else 'no significant difference'}")
display(mean_ranks.to_frame().round(2))

wx_rows = []
for a, b_ in pairs:
    try:
        stat, p = stats.wilcoxon(cv_acc[a], cv_acc[b_])
    except ValueError:
        stat, p = np.nan, 1.0
    wx_rows.append({"Model A": a, "Model B": b_, "Mean diff (A-B)": np.mean(cv_acc[a] - cv_acc[b_]), "W": stat, "p": p})
wx_df = pd.DataFrame(wx_rows)
wx_df["p (Holm)"] = multipletests(wx_df["p"], method="holm")[1]
wx_df["Significance"] = wx_df["p (Holm)"].map(stars)
display(wx_df.round(5)); save_table(wx_df, "Table12a_wilcoxon", index=False, caption="Table 12a. Pairwise Wilcoxon signed-rank test on CV fold accuracies (Holm-corrected).")
save_table(mean_ranks.to_frame(), "Table12b_friedman_mean_ranks", caption="Table 12b. Friedman test mean ranks per model.")

## 10. Learning and Validation Curves

In [ ]:
# --- 10a. Learning curves (training set, subject-independent CV) ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
for ax, name in zip(axes.ravel(), MODEL_NAMES):
    est = clone(MODELS[name]).set_params(**best_params[name])
    sizes, tr, va = learning_curve(est, X_train, y_train, groups=g_train, cv=cv_inner, train_sizes=np.linspace(0.1, 1.0, 6),
                                   scoring="accuracy", n_jobs=-1, shuffle=True, random_state=SEED)
    ax.plot(sizes, tr.mean(1), "o--", label="Train"); ax.fill_between(sizes, tr.mean(1) - tr.std(1), tr.mean(1) + tr.std(1), alpha=.15)
    ax.plot(sizes, va.mean(1), "s-", label="Validation (CV)"); ax.fill_between(sizes, va.mean(1) - va.std(1), va.mean(1) + va.std(1), alpha=.15)
    ax.set_title(name); ax.set_xlabel("Number of training samples"); ax.set_ylabel("Accuracy"); ax.legend(loc="lower right")
plt.tight_layout(); savefig("Figure18_learning_curves", caption="Figure 18. Learning curves (subject-grouped CV) for all models.")

In [ ]:
# --- 10b. Validation curves (single hyperparameter) ---
VC = {"SVM (Linear)":  ("clf__C", [0.001, 0.01, 0.1, 1, 10, 100], True),
      "SVM (RBF)":     ("clf__C", [0.1, 1, 10, 100, 1000], True),
      "KNN":           ("clf__n_neighbors", [1, 3, 5, 7, 9, 11, 15, 21, 31, 51], False),
      "Decision Tree": ("clf__max_depth", [2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 30], False)}
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, name in zip(axes.ravel(), MODEL_NAMES):
    pname, prange, logx = VC[name]
    est = clone(MODELS[name]).set_params(**best_params[name])
    tr, va = validation_curve(est, X_train, y_train, param_name=pname, param_range=prange, groups=g_train,
                              cv=cv_inner, scoring="accuracy", n_jobs=-1)
    plot = ax.semilogx if logx else ax.plot
    plot(prange, tr.mean(1), "o--", label="Train"); plot(prange, va.mean(1), "s-", label="Validation (CV)")
    ax.fill_between(prange, va.mean(1) - va.std(1), va.mean(1) + va.std(1), alpha=.15)
    ax.set_title(f"{name}: {pname.replace('clf__', '')}"); ax.set_xlabel(pname.replace("clf__", "")); ax.set_ylabel("Accuracy"); ax.legend()
plt.tight_layout(); savefig("Figure19_validation_curves", caption="Figure 19. Validation curves (subject-grouped CV) for a key hyperparameter of each model.")

## 11. Dimensionality Reduction (PCA) Experiment
How do accuracy and training time change as the number of features is reduced? (Best hyperparameters held fixed.)

In [ ]:
n_list = [5, 10, 20, 50, 100, 200, None]
pca_rows = []
for name in MODEL_NAMES:
    for nc in n_list:
        pipe = Pipeline([("scaler", StandardScaler()), ("pca", PCA(n_components=nc, random_state=SEED) if nc else "passthrough"),
                         ("clf", clone(final_models[name].named_steps["clf"]))])
        t0 = time.perf_counter(); pipe.fit(X_train, y_train); ft = time.perf_counter() - t0
        yp = pipe.predict(X_test)
        pca_rows.append({"Model": name, "Num. Components": nc if nc else len(feature_cols),
                         "Accuracy": accuracy_score(y_test, yp), "F1 (macro)": f1_score(y_test, yp, average="macro"), "Training Time (s)": ft})
pca_df = pd.DataFrame(pca_rows)
save_table(pca_df, "Table13_pca_experiment", index=False, caption="Table 13. Accuracy, F1 and training time as a function of the number of PCA components.")
display(pca_df.pivot(index="Num. Components", columns="Model", values="Accuracy")[MODEL_NAMES].round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name in MODEL_NAMES:
    d = pca_df[pca_df.Model == name]
    axes[0].plot(d["Num. Components"], d["Accuracy"], "o-", color=MODEL_COLORS[name], label=name)
    axes[1].plot(d["Num. Components"], d["Training Time (s)"], "o-", color=MODEL_COLORS[name], label=name)
axes[0].set_xscale("log"); axes[0].set_xlabel("Number of PCA components"); axes[0].set_ylabel("Test Accuracy"); axes[0].set_title("PCA vs. Accuracy"); axes[0].legend()
axes[1].set_xscale("log"); axes[1].set_yscale("log"); axes[1].set_xlabel("Number of PCA components"); axes[1].set_ylabel("Training Time (s)"); axes[1].set_title("PCA vs. Training Time")
plt.tight_layout(); savefig("Figure20_pca_experiment", caption="Figure 20. Effect of the number of PCA components on test accuracy and training time.")

## 12. Feature Analysis and Ablation

In [ ]:
# --- 12a. Decision Tree feature importances ---
dt_clf = final_models["Decision Tree"].named_steps["clf"]
imp = pd.Series(dt_clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
top20 = imp.head(20)
plt.figure(figsize=(9, 7))
sns.barplot(x=top20.values, y=top20.index, color="tab:blue"); plt.xlabel("Importance (Gini/entropy decrease)"); plt.ylabel("")
plt.title("Decision Tree -- Top 20 Most Important Features"); plt.yticks(fontsize=8)
savefig("Figure21_decision_tree_importances", caption="Figure 21. Top 20 most important features according to the Decision Tree.")
save_table(top20.rename("Importance").to_frame(), "Table14a_top20_features", caption="Table 14a. Top 20 most important features (Decision Tree).")

imp_group = pd.DataFrame({"sensor": meta.sensor.values, "domain": meta.domain.values, "importance": dt_clf.feature_importances_})
grp_tab = imp_group.pivot_table(index="sensor", columns="domain", values="importance", aggfunc="sum", fill_value=0)
plt.figure(figsize=(6, 3.8))
sns.heatmap(grp_tab, annot=True, fmt=".3f", cmap="OrRd"); plt.title("Total Importance -- Sensor x Domain"); plt.ylabel("Sensor"); plt.xlabel("Domain")
savefig("Figure22_group_importances", caption="Figure 22. Total feature importance aggregated by sensor and domain.")

# --- 12b. Feature-group ablation ---
subsets = {"All features": np.ones(len(feature_cols), bool),
           "Accelerometer only (Acc)": (meta.sensor == "Acc").to_numpy(),
           "Gyroscope only (Gyro)": (meta.sensor == "Gyro").to_numpy(),
           "Time domain only": (meta.domain == "Time").to_numpy(),
           "Frequency domain only": (meta.domain == "Frequency").to_numpy()}
ab_rows = []
for sname, mask in subsets.items():
    for name in MODEL_NAMES:
        est = clone(MODELS[name]).set_params(**best_params[name]).fit(X_train[:, mask], y_train)
        yp = est.predict(X_test[:, mask])
        ab_rows.append({"Feature Subset": sname, "Num. Features": int(mask.sum()), "Model": name,
                        "Accuracy": accuracy_score(y_test, yp), "F1 (macro)": f1_score(y_test, yp, average="macro")})
ab_df = pd.DataFrame(ab_rows)
save_table(ab_df, "Table14b_feature_ablation", index=False, caption="Table 14b. Feature-group ablation: accuracy and F1 by feature subset and model.")
ab_piv = ab_df.pivot(index="Feature Subset", columns="Model", values="Accuracy").loc[list(subsets), MODEL_NAMES]
ab_piv.insert(0, "n", [int(m.sum()) for m in subsets.values()])
display(ab_piv.round(4))

plt.figure(figsize=(11, 5))
sns.barplot(data=ab_df, x="Feature Subset", y="Accuracy", hue="Model", palette=MODEL_COLORS)
plt.ylim(max(0, ab_df.Accuracy.min() - 0.05), 1.0); plt.xticks(rotation=20, ha="right"); plt.title("Feature-Group Ablation (Test Accuracy)"); plt.legend(ncol=2, fontsize=8)
savefig("Figure23_feature_ablation", caption="Figure 23. Test accuracy under feature-group ablation.")

## 13. Noise Robustness
Gaussian noise scaled to `sigma` times each feature's training-set standard deviation is added to the test
features (an approximate representation of sensor noise in feature space, not a simulation of noise on the raw signal).

In [ ]:
sigmas = [0, 0.05, 0.1, 0.25, 0.5, 1.0]
feat_std = X_train.std(axis=0)
rng = np.random.default_rng(SEED)
noise_rows = []
for s in sigmas:
    Xn = X_test + rng.standard_normal(X_test.shape) * feat_std * s   # same noise realization applied to all models
    for name in MODEL_NAMES:
        yp = final_models[name].predict(Xn)
        noise_rows.append({"sigma": s, "Model": name, "Accuracy": accuracy_score(y_test, yp), "F1 (macro)": f1_score(y_test, yp, average="macro")})
noise_df = pd.DataFrame(noise_rows); save_table(noise_df, "Table15_noise_robustness", index=False,
                                                  caption="Table 15. Test accuracy and F1 under increasing feature-space Gaussian noise.")
display(noise_df.pivot(index="sigma", columns="Model", values="Accuracy")[MODEL_NAMES].round(4))

plt.figure(figsize=(8, 5))
for name in MODEL_NAMES:
    d = noise_df[noise_df.Model == name]; plt.plot(d["sigma"], d["Accuracy"], "o-", color=MODEL_COLORS[name], label=name)
plt.xlabel("Noise level sigma (x feature std.)"); plt.ylabel("Test Accuracy"); plt.title("Noise Robustness"); plt.legend()
savefig("Figure24_noise_robustness", caption="Figure 24. Test accuracy as a function of injected feature-space noise level.")

## 14. Computational Complexity and Efficiency

In [ ]:
cx_rows = []
for name in MODEL_NAMES:
    clf = final_models[name].named_steps["clf"]
    row = {"Model": name, "Training Time (s)": results_df.loc[name, "Training Time (s)"],
           "Inference (ms/sample)": results_df.loc[name, "Inference Time (ms/sample)"], "Test Accuracy": results_df.loc[name, "Accuracy"]}
    if isinstance(clf, SVC):
        row["Model Size Indicator"] = f"{int(clf.n_support_.sum())} support vectors"
    elif isinstance(clf, KNeighborsClassifier):
        row["Model Size Indicator"] = f"{len(X_train)} stored samples (lazy learner)"
    else:
        row["Model Size Indicator"] = f"depth={clf.get_depth()}, leaves={clf.get_n_leaves()}, nodes={clf.tree_.node_count}"
    cx_rows.append(row)
cx_df = pd.DataFrame(cx_rows).set_index("Model"); display(cx_df); save_table(cx_df, "Table16_complexity_efficiency",
                                                                                caption="Table 16. Training time, inference speed, model size and test accuracy per model.")

fig, ax = plt.subplots(figsize=(8.5, 5.5))
for name in MODEL_NAMES:
    ax.scatter(cx_df.loc[name, "Inference (ms/sample)"], cx_df.loc[name, "Test Accuracy"], s=80 + 40 * cx_df.loc[name, "Training Time (s)"] ** 0.5,
               color=MODEL_COLORS[name], edgecolor="k", label=name)
    ax.annotate(name, (cx_df.loc[name, "Inference (ms/sample)"], cx_df.loc[name, "Test Accuracy"]), textcoords="offset points", xytext=(8, 6), fontsize=9)
ax.set_xscale("log"); ax.set_xlabel("Inference time (ms/sample, log)"); ax.set_ylabel("Test Accuracy")
ax.set_title("Accuracy-Speed Trade-off (bubble size proportional to training time)")
savefig("Figure25_accuracy_speed", caption="Figure 25. Accuracy-speed trade-off across models (bubble size proportional to training time).")

## 15. Summary Table and Download of All Outputs

In [ ]:
master = results_df[["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)", "MCC", "ROC-AUC (OvR)",
                     "Training Time (s)", "Inference Time (ms/sample)"]].copy()
master[f"CV Accuracy ({N_OUTER}-fold)"] = [cv_summary.loc[n, "Accuracy"] for n in master.index]
master["Rank (F1)"] = master["F1 (macro)"].rank(ascending=False).astype(int)
master = master.sort_values("Rank (F1)")
display(master.round(4)); save_table(master, "Table17_master_summary", caption="Table 17. Master summary table across all evaluation metrics.")

with open(OUT_DIR / "config_and_environment.json", "w", encoding="utf-8") as f:
    json.dump({"env": env_info, "best_params": {k: {p: str(v) for p, v in d.items()} for k, d in best_params.items()},
               "FAST_MODE": FAST_MODE, "N_OUTER": N_OUTER, "N_BOOT": N_BOOT}, f, indent=2, ensure_ascii=False)

# Manifest of every figure/table produced, for traceability in the manuscript
with open(OUT_DIR / "figures_manifest.json", "w", encoding="utf-8") as f:
    json.dump(FIGURE_LOG, f, indent=2, ensure_ascii=False)
with open(OUT_DIR / "tables_manifest.json", "w", encoding="utf-8") as f:
    json.dump(TABLE_LOG, f, indent=2, ensure_ascii=False)
print(f"{len(FIGURE_LOG)} figures and {len(TABLE_LOG)} tables logged in outputs/figures_manifest.json and outputs/tables_manifest.json")

zip_path = shutil.make_archive("har_experiment_outputs", "zip", OUT_DIR)
print("Archive created:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab; the archive is in the working directory.")